# FEMI-SOURCE-BINDING-JEV-1 — Causal intervention & decision-model selection

**New FEVEROUS cases, frozen 360-call auditor factorial, 36 prospective Jev predictions, optional local d1 selector.**

This notebook does **not** rerun old FEMI experiments and does not conduct d1 activation steering. It estimates the causal effect of **assigned prompt treatments** on auditor behavior and evaluates whether a decision model predicts useful behavioral interventions. No paid inference occurs until the clearly labeled `--execute` cells.

**Do not regenerate the manifest once paid calls begin, and do not inspect partial auditor results before the Jev selection is complete.**

## 1 — Upload and unpack the execution packet (offline)

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, os, subprocess, json, hashlib, shutil

bundle = files.upload()
zipname = next((name for name in bundle if name.endswith('FEMI_CAUSAL_SOURCE_JEV_20261108.zip')), None)
assert zipname, 'Upload FEMI_CAUSAL_SOURCE_JEV_20261108.zip'
with zipfile.ZipFile(zipname) as z:
    assert z.testzip() is None, 'Packet ZIP integrity error'
    z.extractall('/content')
root = Path('/content/FEMI_CAUSAL_SOURCE_JEV_20261108')
assert (root / 'src/build.py').is_file()
os.chdir(root)
print('Project:', Path.cwd())

## 2 — Offline software tests (no network and no GPU)

In [ ]:
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],check=True)
print('Synthetic software tests passed. These are NOT experimental findings.')

## 3 — Reuse previously extracted FEVEROUS data, freeze 36 genuinely new claims

In [ ]:
DEV = Path('/content/datasets/feverous/feverous_dev_challenges.jsonl')
WIKI = Path('/content/datasets/feverous/FeverousWikiv1')
assert DEV.is_file(), f'Expected official FEVEROUS dev split at {DEV}'
assert WIKI.is_dir() and list(WIKI.glob('wiki_*.jsonl')), f'Expected extracted FEVEROUS shards at {WIKI}'
run_dir=Path('runs/source_causal')
if (run_dir/'FREEZE.json').exists():
    print('Existing freeze found; NOT rebuilding.')
else:
    assert not run_dir.exists(), 'Partial run folder exists. Inspect it; do not silently rebuild.'
    subprocess.run(['python','src/build.py','--dev',str(DEV),'--wiki',str(WIKI),
                    '--out',str(run_dir),'--n','36','--min-table','12',
                    '--seed','81227'],check=True)
subprocess.run(['python','-c',
                "import sys;sys.path.insert(0,'src');from core import verify; c,t,s,_=verify('runs/source_causal');print('VERIFIED: cases',len(c),'auditor requests',len(t),'selectors',len(s))"],check=True)
print('Freeze:',json.loads((run_dir/'FREEZE.json').read_text()))

## 4 — **Human evidence check** before model calls

In [ ]:
cases=[json.loads(l) for l in (run_dir/'cases_PRIVATE.jsonl').read_text().splitlines()]
assert len(cases)==36
assert len({c['source_uid'] for c in cases})==36
assert len({c['claim'] for c in cases})==36
from collections import Counter
print('Labels:',Counter(c['gold'] for c in cases))
print('Evidence modalities:',Counter(c['metadata']['modality'] for c in cases))
for c in cases[:6]:
    print('\n'+'='*65)
    print(c['case_id'], c['source_uid'], c['gold'])
    print('CLAIM:',c['claim'][:350])
    print('EVIDENCE:',c['trusted'][:900])
print('Do not proceed if sources look unresolved, malformed, or clearly contradictory to labels.')

## 5 — Dry-run both API configurations (no network)

In [ ]:
subprocess.run(['python','src/execute.py','jev','--dir',str(run_dir)],check=True)
subprocess.run(['python','src/execute.py','auditor','--dir',str(run_dir)],check=True)

## 6 — PAID: Jev API checkpoint and prospective predictions **before** auditor responses

In [ ]:
from google.colab import userdata
key=userdata.get('OPENROUTER_API_KEY')
assert isinstance(key,str) and key.strip(), 'Set OPENROUTER_API_KEY in Colab Secrets and enable notebook access'
os.environ['OPENROUTER_API_KEY']=key
del key
# The first two requests validate the Decisions API, not scientific outcomes.
subprocess.run(['python','src/execute.py','jev','--dir',str(run_dir),
                '--execute','--max-new','2'],check=True)
# Complete all remaining frozen Jev requests; resumes by case ID and request hash.
subprocess.run(['python','src/execute.py','jev','--dir',str(run_dir),
                '--execute'],check=True)
print('Jev predictions frozen before any auditor calls.')

## 7 — PAID: GPT-OSS/Groq target auditor on the frozen counterfactual grid

In [ ]:
# This requests GPT-OSS-120B with Groq-only provider and no fallback.
# The first ten calls are a routing/schema checkpoint only. Do not tune the design based on interim outcomes.
subprocess.run(['python','src/execute.py','auditor','--dir',str(run_dir),
                '--execute','--max-new','10'],check=True)
subprocess.run(['python','src/execute.py','auditor','--dir',str(run_dir),
                '--execute'],check=True)
print('Auditor collection finished. No changes to the frozen manifest were made.')

## 8 — Analyze auditor results, Jev selection and baseline heuristics (offline)

In [ ]:
subprocess.run(['python','src/analyze.py','--dir',str(run_dir)],check=True)
report=json.loads((run_dir/'results/INDEPENDENT_RECOMPUTATION.json').read_text())
print('Counts:',report['n_cases'],report['n_auditor_trials'],report['n_jev_selector_trials'])
print('Selection comparison:',json.dumps(report['selection_policies'],indent=2))

## 9 — OPTIONAL GPU: unsteered d1 as a second **intervention predictor**, not an activation-steering intervention

In [ ]:
# OPTIONAL: Only run in a Colab GPU runtime if you want a same-task d1 comparison.
# No local model or package is installed by default; explicitly opt in.
RUN_D1 = False
if RUN_D1:
    subprocess.run(['pip','install','-q','transformers>=5.14','torch','torchvision','pillow'],check=True)
    subprocess.run(['python','src/run_d1_optional.py','--dir',str(run_dir),'--execute'],check=True)
    subprocess.run(['python','src/analyze.py','--dir',str(run_dir)],check=True)
else:
    print('Skipped optional local d1 selector. Jev/auditor experiment unaffected.')

## 10 — Export the **entire frozen experiment and raw replies**, no dataset download required

In [ ]:
assert (run_dir/'results/INDEPENDENT_RECOMPUTATION.json').is_file()
archive=shutil.make_archive('/content/FEMI_SOURCE_BINDING_JEV_1_RESULTS','zip',
      root_dir='/content',base_dir='FEMI_CAUSAL_SOURCE_JEV_20261108')
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
print('Archive:',archive,'MB',round(Path(archive).stat().st_size/1e6,2))
print('SHA-256',hashlib.sha256(Path(archive).read_bytes()).hexdigest())
files.download(archive)